In [724]:
import nflreadpy as nfl
import pandas as pd
import numpy as np

In [725]:
SEASON = 2025
pbp = nfl.load_pbp([SEASON]).to_pandas()
schedule = nfl.load_schedules([SEASON]).to_pandas()

In [726]:
scrimmage_plays = pbp.loc[
    pbp["posteam"].notna()
    & pbp["defteam"].notna()
    & pbp["epa"].notna()
    & pbp["success"].notna()
    & (pbp["pass"].eq(1) | pbp["rush"].eq(1))
    & pbp["qb_kneel"].fillna(0).ne(1)
    & pbp["qb_spike"].fillna(0).ne(1)
    & pbp["two_point_attempt"].fillna(0).ne(1)
].copy()

In [727]:
offense = (
    scrimmage_plays
    .groupby(['game_id', 'posteam'])
    .agg(
        off_epa = ('epa', 'mean'),
        off_success_rate = ('success', 'mean')
    )
    .reset_index()
)

In [728]:
pass_df = (
    scrimmage_plays.loc[scrimmage_plays["pass"].eq(1)]
    .groupby(['game_id', 'posteam'])
    .agg(pass_epa = ('epa', 'mean'))
    .reset_index()
)
rush_df = (
    scrimmage_plays.loc[scrimmage_plays["rush"].eq(1)]
    .groupby(['game_id', 'posteam'])
    .agg(rush_epa = ('epa', 'mean'))
    .reset_index()
)

In [729]:
offense_df = (
    offense
    .merge(pass_df, on=['game_id', 'posteam'], how='left')
    .merge(rush_df, on=['game_id', 'posteam'], how='left')
    .rename(columns={'posteam': 'team'})
)
offense_df.head()

,game_id,team,off_epa,off_success_rate,pass_epa,rush_epa
0,2025_01_ARI_NO,ARI,0.061773,0.424242,0.131530,-0.052630
1,2025_01_ARI_NO,NO,-0.057140,0.424658,-0.014034,-0.157067
2,2025_01_BAL_BUF,BAL,0.394435,0.509804,0.530099,0.282997
3,2025_01_BAL_BUF,BUF,0.302612,0.512821,0.457560,-0.046021
4,2025_01_CAR_JAX,CAR,-0.176198,0.365079,-0.214952,-0.092878


In [730]:
defense_df = (
    scrimmage_plays
    .groupby(['game_id', 'defteam'])
    .agg(
        def_epa_allowed = ('epa', 'mean'),
        def_success_rate_allowed = ('success', 'mean'),
    )
    .reset_index()
    .rename(columns={'defteam': 'team'})
)
defense_df.head()

,game_id,team,def_epa_allowed,def_success_rate_allowed
0,2025_01_ARI_NO,ARI,-0.057140,0.424658
1,2025_01_ARI_NO,NO,0.061773,0.424242
2,2025_01_BAL_BUF,BAL,0.302612,0.512821
3,2025_01_BAL_BUF,BUF,0.394435,0.509804
4,2025_01_CAR_JAX,CAR,0.103053,0.460317


In [731]:
schedule_df = schedule.loc[
    schedule['home_score'].notna()
    & schedule['away_score'].notna(),
    ['game_id', 'season', 'week', 'gameday', 'home_team', 'away_team', 'home_score', 'away_score', 'home_rest', 'away_rest', 'location'],
].copy()

In [732]:
schedule_df['home_win'] = np.where(
    schedule_df['home_score'].eq(schedule_df['away_score']), np.nan,
    schedule_df['home_score'].gt(schedule_df['away_score']).astype(int)
)

In [733]:
schedule_df['neutral_site'] = (schedule_df['location'].astype(str).str.lower().eq('neutral')).astype(int)

In [734]:
home_rows = schedule_df[['season', 'week', 'gameday', 'game_id', 'neutral_site']].copy()

home_rows['team'] = schedule_df['home_team']
home_rows['opponent'] = schedule_df['away_team']
home_rows['points_scored'] = schedule_df['home_score']
home_rows['points_allowed'] = schedule_df['away_score']
home_rows['rest_days'] = schedule_df['home_rest']
home_rows['is_home'] = 1

away_rows = schedule_df[['season', 'week', 'gameday', 'game_id', 'neutral_site']].copy()

away_rows['team'] = schedule_df['away_team']
away_rows['opponent'] = schedule_df['home_team']
away_rows['points_scored'] = schedule_df['away_score']
away_rows['points_allowed'] = schedule_df['home_score']
away_rows['rest_days'] = schedule_df['away_rest']
away_rows['is_home'] = 0

In [735]:
def add_team_result(df):
    df = df.copy()

    df['win'] = np.select(
        [
            df['points_scored'].gt(df['points_allowed']),
            df['points_scored'].lt(df['points_allowed'])
        ],
        [1.0, 0.0],
        default=0.5,
    )
    df['home_field'] = (
    df['is_home'].eq(1) 
    & df['neutral_site'].eq(0)
    ).astype(int)

    return df

home_rows = add_team_result(home_rows)
away_rows = add_team_result(away_rows)

In [736]:
schedule_df = pd.concat([home_rows, away_rows], ignore_index=True)

In [737]:
team_games_df = (
    schedule_df
    .merge(
        offense_df, 
        on=['game_id', 'team'], 
        how='left',
    )
    .merge(
        defense_df, 
        on=['game_id', 'team'], 
        how='left',
    )    
)

In [738]:
team_games_df = (
    team_games_df
    .sort_values(['team', 'gameday', 'game_id'])
    .reset_index(drop=True)
)

In [739]:
team_games_df = team_games_df[[
    'season',
    'week',
    'gameday',  
    'game_id',
    'team',
    'opponent',
    'is_home',
    'neutral_site',
    'home_field',
    'rest_days',
    'off_epa',
    'off_success_rate',
    'pass_epa',
    'rush_epa',
    'def_epa_allowed',
    'def_success_rate_allowed',
    'points_scored',
    'points_allowed',
    'win',
]]

In [740]:
team_games_df.head()

,season,week,gameday,game_id,team,opponent,is_home,neutral_site,home_field,rest_days,off_epa,off_success_rate,pass_epa,rush_epa,def_epa_allowed,def_success_rate_allowed,points_scored,points_allowed,win
0,2025,1,2025-09-07,2025_01_ARI_NO,ARI,NO,0,0,0,7,0.061773,0.424242,0.131530,-0.052630,-0.057140,0.424658,20,13,1.0
1,2025,2,2025-09-14,2025_02_CAR_ARI,ARI,CAR,1,0,1,7,0.085936,0.425926,0.251884,-0.196177,0.060806,0.487805,27,22,1.0
2,2025,3,2025-09-21,2025_03_ARI_SF,ARI,SF,0,0,0,7,0.045186,0.393939,0.145420,-0.119197,-0.091097,0.507937,15,16,0.0
3,2025,4,2025-09-25,2025_04_SEA_ARI,ARI,SEA,1,0,1,4,-0.060240,0.388060,-0.025498,-0.204554,0.106473,0.439394,20,23,0.0
4,2025,5,2025-10-05,2025_05_TEN_ARI,ARI,TEN,1,0,1,10,-0.059299,0.367647,0.129428,-0.328909,-0.025627,0.460317,21,22,0.0


In [741]:
team_games_df.loc[
    team_games_df["game_id"] == "2025_01_BAL_BUF"
]

,season,week,gameday,game_id,team,opponent,is_home,neutral_site,home_field,rest_days,off_epa,off_success_rate,pass_epa,rush_epa,def_epa_allowed,def_success_rate_allowed,points_scored,points_allowed,win
34,2025,1,2025-09-07,2025_01_BAL_BUF,BAL,BUF,0,0,0,7,0.394435,0.509804,0.530099,0.282997,0.302612,0.512821,40,41,0.0
51,2025,1,2025-09-07,2025_01_BAL_BUF,BUF,BAL,1,0,1,7,0.302612,0.512821,0.457560,-0.046021,0.394435,0.509804,41,40,1.0
